In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from sklearn.cluster import HDBSCAN
import numpy as np
import collections
import json


print("Loading tokenizer and model...")
xlstm_config = AutoConfig.from_pretrained("NX-AI/xLSTM-7b")
xlstm_config.step_kernel = "native"
xlstm_config.chunkwise_kernel = "chunkwise--native_autograd"
xlstm_config.sequence_kernel = "native_sequence__native"

model = AutoModelForCausalLM.from_pretrained(
    "NX-AI/xLSTM-7b",
    config=xlstm_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("NX-AI/xLSTM-7b")



In [ ]:
print("Reading text...")
with open("big_text_prompt.txt", "r", encoding="utf-8") as f:
    prompt = f.read().strip()

inputs = tokenizer(prompt, return_tensors="pt")['input_ids'].to(model.device)
bos_id = tokenizer.bos_token_id
bos_tensor = torch.tensor([[bos_id]], device=model.device, dtype=inputs.dtype)
tokens_with_bos = torch.cat([bos_tensor, inputs], dim=1)

input_tokens = [tokenizer.decode(t) for t in tokens_with_bos[0]]
seq_len = len(input_tokens)
print(f"Sequence length: {seq_len} tokens")

print("Running forward pass...")
with torch.no_grad():
    outputs = model(tokens_with_bos, output_hidden_states=True)

hidden_states = outputs.hidden_states # Tuple of 33 tensors of (1, seq_len, 4096)

results = {}

print("Running HDBSCAN per layer...")
# Iterate over each block's output
# hidden_states[0] is embeddings, hidden_states[1] is output of block 0, etc.
for layer_idx, h_state in enumerate(hidden_states):
    # Shape: (seq_len, 4096)
    X = h_state[0].float().cpu()
    
    # Normalize for cosine similarity equivalent using euclidean metric
    X_norm = F.normalize(X, p=2, dim=1).numpy()
    
    # We use min_cluster_size=3 since seq_len is around 170
    clusterer = HDBSCAN(min_cluster_size=3, metric='euclidean', cluster_selection_epsilon=0.1)
    labels = clusterer.fit_predict(X_norm)
    
    # Analyze clusters
    unique_labels = set(labels)
    n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    # Group tokens by cluster
    clusters_dict = collections.defaultdict(list)
    for token, label in zip(input_tokens, labels):
        clusters_dict[int(label)].append(token.strip())
        
    layer_res = {
        "n_clusters": n_clusters,
        "n_noise_points": n_noise,
        "clusters": clusters_dict
    }
    results[layer_idx] = layer_res
    
    print(f"Layer {layer_idx}: {n_clusters} clusters, {n_noise} noise points")

with open("hdbscan_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
    
print("Experiment finished. Results saved to hdbscan_results.json")

